# 4 · Normalization (robust scaling, fit on train only)

Median/IQR scaling per column, per TF. Scaler fit on **train split only**, then applied to val/test —
no leakage. Robust (not z-score) since forex has fat-tailed spikes that would skew mean/std.

Columns skipped (already bounded/categorical, scaling would break their meaning):
- `datetime`
- binary flags: `ema_cross_up`, `ema_cross_down`
- sign columns: `ema_regime`, `ema_trend`

Scaler params (median, IQR per column) saved per TF so the same transform can be replayed in
paper/live trading.

In [1]:
import sys
import json
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd

from config.settings import ROOT

DATA_SPLITS = ROOT / "data" / "splits"
DATA_NORM = ROOT / "data" / "normalized"
DATA_NORM.mkdir(parents=True, exist_ok=True)
SCALER_DIR = ROOT / "config" / "scalers"
SCALER_DIR.mkdir(parents=True, exist_ok=True)

SYMBOL = "GBPUSD"
TIMEFRAMES = ["D1", "H4", "H1", "M15"]
SPLITS = ["train", "val", "test"]

SKIP_COLS = {
    "datetime",
    "ema_cross_up", "ema_cross_down",
    "ema_regime", "ema_trend",
}

## Fit scaler on train (median / IQR per column)

In [2]:
def fit_scaler(train_df: pd.DataFrame, cols: list[str]) -> dict:
    scaler = {}
    for col in cols:
        median = train_df[col].median()
        q75, q25 = train_df[col].quantile([0.75, 0.25])
        iqr = q75 - q25
        if iqr == 0:
            iqr = 1.0  # constant column guard — avoid div by zero
        scaler[col] = {"median": float(median), "iqr": float(iqr)}
    return scaler


def apply_scaler(df: pd.DataFrame, scaler: dict, cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in cols:
        out[col] = (df[col] - scaler[col]["median"]) / scaler[col]["iqr"]
    return out


scalers = {}
for tf in TIMEFRAMES:
    train_df = pd.read_parquet(DATA_SPLITS / f"{SYMBOL}_{tf}_train.parquet")
    scale_cols = [c for c in train_df.columns if c not in SKIP_COLS]

    scaler = fit_scaler(train_df, scale_cols)
    scalers[tf] = {"cols": scale_cols, "params": scaler}

    path = SCALER_DIR / f"{SYMBOL}_{tf}_scaler.json"
    path.write_text(json.dumps(scaler, indent=2))
    print(f"[FIT] {tf}: {len(scale_cols)} cols scaled, {len(SKIP_COLS & set(train_df.columns))} skipped -> {path.relative_to(ROOT)}")

[FIT] D1: 5 cols scaled, 2 skipped -> config/scalers/GBPUSD_D1_scaler.json
[FIT] H4: 6 cols scaled, 2 skipped -> config/scalers/GBPUSD_H4_scaler.json
[FIT] H1: 5 cols scaled, 3 skipped -> config/scalers/GBPUSD_H1_scaler.json
[FIT] M15: 9 cols scaled, 3 skipped -> config/scalers/GBPUSD_M15_scaler.json


## Apply to train / val / test, save

In [3]:
for tf in TIMEFRAMES:
    scale_cols = scalers[tf]["cols"]
    scaler = scalers[tf]["params"]

    for split in SPLITS:
        df = pd.read_parquet(DATA_SPLITS / f"{SYMBOL}_{tf}_{split}.parquet")
        normed = apply_scaler(df, scaler, scale_cols)

        path = DATA_NORM / f"{SYMBOL}_{tf}_{split}.parquet"
        normed.to_parquet(path, index=False)
        print(f"[SAVE] {path.relative_to(ROOT)}: {len(normed)} rows")
    print()

[SAVE] data/normalized/GBPUSD_D1_train.parquet: 2603 rows
[SAVE] data/normalized/GBPUSD_D1_val.parquet: 520 rows
[SAVE] data/normalized/GBPUSD_D1_test.parquet: 660 rows

[SAVE] data/normalized/GBPUSD_H4_train.parquet: 15608 rows
[SAVE] data/normalized/GBPUSD_H4_val.parquet: 3120 rows
[SAVE] data/normalized/GBPUSD_H4_test.parquet: 3952 rows

[SAVE] data/normalized/GBPUSD_H1_train.parquet: 62219 rows
[SAVE] data/normalized/GBPUSD_H1_val.parquet: 12470 rows
[SAVE] data/normalized/GBPUSD_H1_test.parquet: 15737 rows

[SAVE] data/normalized/GBPUSD_M15_train.parquet: 245307 rows
[SAVE] data/normalized/GBPUSD_M15_val.parquet: 49179 rows
[SAVE] data/normalized/GBPUSD_M15_test.parquet: 62013 rows



## Sanity check — train median ≈ 0, IQR ≈ 1; val/test may drift (expected, uses train params)

In [4]:
for tf in TIMEFRAMES:
    scale_cols = scalers[tf]["cols"]
    for split in SPLITS:
        df = pd.read_parquet(DATA_NORM / f"{SYMBOL}_{tf}_{split}.parquet")
        med = df[scale_cols].median()
        print(f"{tf}/{split}: median range [{med.min():.3f}, {med.max():.3f}]")
    print()

D1/train: median range [-0.000, 0.000]
D1/val: median range [-0.559, 0.244]
D1/test: median range [-0.744, 0.250]

H4/train: median range [0.000, 0.000]
H4/val: median range [-0.584, -0.017]
H4/test: median range [-0.283, 0.018]

H1/train: median range [0.000, 0.000]
H1/val: median range [-0.581, 0.464]
H1/test: median range [-0.492, 0.044]

M15/train: median range [-0.000, 0.000]
M15/val: median range [-0.580, 0.317]
M15/test: median range [-0.315, 0.021]

